# aplay+ Build Notebook
Builds **aplay+** (bit-perfect audio player) with Intel `icx` compiler and creates RPM package.

In [ ]:
%%bash
# 1. Setup Intel oneAPI repository (Ubuntu 24.04 compatible)
sudo apt-get update -qq
sudo apt-get install -y gpg-agent wget ca-certificates software-properties-common

wget -qO- https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | \
  gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null

echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | \
  sudo tee /etc/apt/sources.list.d/oneAPI.list

sudo apt-get update -qq

In [ ]:
%%bash
# 2. Install Intel compiler (lightweight version)
sudo apt-get install -y intel-oneapi-compiler-dpcpp-cpp-and-cpp-classic

In [ ]:
%%bash
# 3. Verify compiler
source /opt/intel/oneapi/setvars.sh --force
icx --version
icpx --version

In [ ]:
%%bash
# 4. Install build dependencies
sudo apt-get install -y libasound2-dev rpm build-essential git

In [ ]:
%%bash
# 5. Build aplay+
source /opt/intel/oneapi/setvars.sh --force
cd /content

rm -rf aplay* aplay+
git clone https://github.com/yui0/aplay-.git
cd aplay-

echo "Building with CC=icx"
CC=icx make -j2

ls -lh aplay+
file aplay+

In [ ]:
%%bash
# 6. Package as RPM
cd /content
mv aplay- aplay+-1.3
tar cvjf aplay+-1.3.tar.bz2 aplay+-1.3

rpmbuild -ta aplay+-1.3.tar.bz2 --nodeps --define "_topdir /root/rpmbuild"

ls -lh /root/rpmbuild/RPMS/x86_64/

In [ ]:
# 7. Upload the RPM (or download manually)
!ls -lh /root/rpmbuild/RPMS/x86_64/aplay+*.rpm
!curl -T /root/rpmbuild/RPMS/x86_64/aplay+*.rpm temp.sh